In [1]:
import pandas as pd
import sqlite3
import numpy as np
import warnings
warnings.filterwarnings('ignore')

db_path = r'C:\Users\beril.oztan\Desktop\rossmann-demand-forecast\data\processed\rossmann.db'
conn = sqlite3.connect(db_path)

def query(sql):
    return pd.read_sql_query(sql, conn)

print("Bağlantı tamam!")

Bağlantı tamam!


In [2]:
df = query("""
    SELECT *
    FROM sales_master
    ORDER BY Store, Date
""")

df['Date'] = pd.to_datetime(df['Date'])

print(df.shape)
df.head()

(844338, 12)


,Store,Date,DayOfWeek,Sales,Customers,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2
0,1,2013-01-02,3,5530,668,0,0,1,c,a,1270.0,0
1,1,2013-01-03,4,4327,578,0,0,1,c,a,1270.0,0
2,1,2013-01-04,5,4486,619,0,0,1,c,a,1270.0,0
3,1,2013-01-05,6,4997,635,0,0,1,c,a,1270.0,0
4,1,2013-01-07,1,7176,785,1,0,1,c,a,1270.0,0


In [3]:
# Date sütunundan yeni özellikler türetiyoruz
df['Year'] = df['Date'].dt.year          # Yıl: 2013, 2014, 2015
df['Month'] = df['Date'].dt.month        # Ay: 1-12
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)  # Haftanın numarası: 1-52
df['Quarter'] = df['Date'].dt.quarter    # Çeyrek: 1-4
df['DayOfYear'] = df['Date'].dt.dayofyear  # Yılın kaçıncı günü: 1-365
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)  # Hafta sonu mu? 1/0
df['IsDecember'] = (df['Month'] == 12).astype(int)  # Aralık ayı mı? EDA'da gördük, kritik

print("Yeni sütunlar:", ['Year','Month','Week','Quarter','DayOfYear','IsWeekend','IsDecember'])
df[['Date','Year','Month','Week','IsWeekend','IsDecember']].head()

Yeni sütunlar: ['Year', 'Month', 'Week', 'Quarter', 'DayOfYear', 'IsWeekend', 'IsDecember']


,Date,Year,Month,Week,IsWeekend,IsDecember
0,2013-01-02,2013,1,1,0,0
1,2013-01-03,2013,1,1,0,0
2,2013-01-04,2013,1,1,0,0
3,2013-01-05,2013,1,1,1,0
4,2013-01-07,2013,1,2,0,0


In [4]:
# Her mağaza için ayrı ayrı lag hesaplıyoruz
# groupby(Store) → mağazaları karıştırmamak için
# shift(7) → 7 satır yukarısındaki değeri al, yani 7 gün önceki satış

df = df.sort_values(['Store', 'Date'])  # Sıralı olması şart

df['lag_7'] = df.groupby('Store')['Sales'].shift(7)    # 7 gün önceki satış
df['lag_14'] = df.groupby('Store')['Sales'].shift(14)  # 14 gün önceki satış
df['lag_30'] = df.groupby('Store')['Sales'].shift(30)  # 30 gün önceki satış

# Rolling mean → son N günün ortalaması
# min_periods=1 → yeterli geçmiş yoksa bile hesapla
df['rolling_mean_7'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)
df['rolling_mean_30'] = df.groupby('Store')['Sales'].transform(
    lambda x: x.shift(1).rolling(30, min_periods=1).mean()
)

print("Yeni sütunlar: lag_7, lag_14, lag_30, rolling_mean_7, rolling_mean_30")
df[['Store', 'Date', 'Sales', 'lag_7', 'rolling_mean_7']].head(10)

Yeni sütunlar: lag_7, lag_14, lag_30, rolling_mean_7, rolling_mean_30


,Store,Date,Sales,lag_7,rolling_mean_7
0,1,2013-01-02,5530,NaN,NaN
1,1,2013-01-03,4327,NaN,5530.000000
2,1,2013-01-04,4486,NaN,4928.500000
3,1,2013-01-05,4997,NaN,4781.000000
4,1,2013-01-07,7176,NaN,4835.000000
5,1,2013-01-08,5580,NaN,5303.200000
6,1,2013-01-09,5471,NaN,5349.333333
7,1,2013-01-10,4892,5530.0,5366.714286
8,1,2013-01-11,4881,4327.0,5275.571429
9,1,2013-01-12,4952,4486.0,5354.714286


In [5]:
# Lag feature'lar üretince ilk 30 günde NaN oluştu
# Çünkü 30 gün öncesi yok — bu satırları atıyoruz

df = df.dropna(subset=['lag_7', 'lag_14', 'lag_30'])

print(f"Temizlenen satır sayısı: {844338 - len(df)}")
print(f"Kalan satır sayısı: {len(df)}")
df.shape

Temizlenen satır sayısı: 33450
Kalan satır sayısı: 810888


(810888, 24)

In [8]:
# Temizlenmiş ve zenginleştirilmiş veriyi veritabanına y# Temizlenmiş ve zenginleştirilmiş veriyi veritabanına yazıyoruz
# if_exists='replace' → tablo varsa sil, yeniden oluştur
# index=False → DataFrame'in index'ini sütun olarak yazma

conn2 = sqlite3.connect(r'C:\Users\beril.oztan\Desktop\rossmann-demand-forecast\data\processed\rossmann.db')

df.to_sql('sales_features', conn2, if_exists='replace', index=False)

print(f"sales_features tablosu kaydedildi: {df.shape[0]} satır, {df.shape[1]} sütun")
conn2.close()

conn2 = sqlite3.connect(r'C:\Users\beril.oztan\Desktop\rossmann-demand-forecast\data\processed\rossmann.db')

df.to_sql('sales_features', conn2, if_exists='replace', index=False)

print(f"sales_features tablosu kaydedildi: {df.shape[0]} satır, {df.shape[1]} sütun")
conn2.close()

sales_features tablosu kaydedildi: 810888 satır, 24 sütun
sales_features tablosu kaydedildi: 810888 satır, 24 sütun
